use motionclip_py310_v2 env

In [1]:
if "_magic_done" not in globals(): # prevent multiple run
    %load_ext autoreload
    %autoreload 2
    %cd ./MotionCLIP
    _magic_done = True

/home/asad/workspace/DomainProject/changeDomain/notebooks/pose/4.motion_clip/MotionCLIP


In [2]:
import sys
sys.path.append('.')

import os
import clip
import torch
import joblib
import numpy as np
from tqdm import tqdm
from IPython.display import Video

# MotionClip
from src.parser.visualize import parser
from src.utils.misc import load_model_wo_clip
from src.datasets.get_dataset import get_datasets
import src.utils.rotation_conversions as geometry
from src.models.get_model import get_model as get_gen_model

In [3]:
sys.argv = [
    "notebook",  # dummy script name
    "./exps/paper-model/checkpoint_0100.pth.tar",
    "--input_file", "./assets/paper_edits.csv",
]


parameters, folder, checkpointname, epoch = parser()

In [4]:
clip_model, clip_preprocess = clip.load("ViT-B/32", device=parameters['device'], jit=False)  # Must set jit=False for training
clip.model.convert_weights(clip_model)  # Actually this line is unnecessary since clip by default already on float16

if parameters.get('clip_training', '') == '':
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False

split='test'
# split='all'  # need more memory
datasets = get_datasets(parameters, clip_preprocess, split)
model = get_gen_model(parameters, clip_model)

print("Restore weights..")
checkpointpath = os.path.join(folder, checkpointname)
state_dict = torch.load(checkpointpath, map_location=parameters["device"])
load_model_wo_clip(model, state_dict)

datapath used by amass is [./data/amass_db/amass_30fps_test.pt]
Restore weights..


/home/asad/workspace/anaconda3/envs/motionclip_py310_v2/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/tmp/ipykernel_9045/1347834188.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.seria

In [5]:
def theta_to_inp(thetas, joints3d, translation = True):
    # Parameters (similar to before)
    pose_rep = "rot6d"
    
    glob = True
    num_frames = -1  # Take all frames
    max_len = -1

    # For the first sample (index 0)

    nframes = thetas.shape[0]

    # Determine frame_ix
    if num_frames == -1 and (max_len == -1 or nframes <= max_len):
        frame_ix = np.arange(nframes)
    else:
        # Add logic if needed, but for simplicity, take all
        pass

    # Load rotvec
    pose = thetas[frame_ix].reshape(-1, 24, 3)  # 72 / 3 = 24 (23 joints + global)

    if not glob:
        pose = pose[:, 1:, :]


    # To rot6d
    ret = geometry.matrix_to_rotation_6d(geometry.axis_angle_to_matrix(pose))

    # Translation from joints3d
    padded_tr = torch.zeros((ret.shape[0], ret.shape[2]), dtype=ret.dtype)
    if translation:
        joints3D = joints3d[frame_ix]
        joints3D = joints3D - joints3D[0, 0, :]
        ret_tr = torch.from_numpy(joints3D[:, 0, :])  # root joint

        padded_tr[:, :3] = ret_tr
    ret = torch.cat((ret, padded_tr[:, None]), 1)

    # Permute to (joints, feats, frames)
    return ret.permute(1, 2, 0).contiguous().float()


# Revert to (nframes, 25, 6)
def inp_to_theta(inp):
    ret = inp.permute(2, 0, 1)

    # Separate rotations and translation
    rot_part = ret[:, :24, :]  # (nframes, 24, 6)

    # Revert rot6d to matrix to axis_angle
    matrix = geometry.rotation_6d_to_matrix(rot_part)
    axis_angle = geometry.matrix_to_axis_angle(matrix)  # (nframes, 24, 3)

    # Flatten to thetas
    return axis_angle.reshape(axis_angle.shape[0], -1)  # (nframes, 72)





In [6]:
def retrieve_motions(inp_list, device):
    retrieved_motions = []
    for inp in inp_list:
        retrieved_motions.append(inp.unsqueeze(0).to(device))
    return torch.cat(retrieved_motions, axis=0)

def encode_motions(model, motions, device):
    return model.encoder({'x': motions,
                        'y': torch.zeros(motions.shape[0], dtype=int, device=device),
                        'mask': model.lengths_to_mask(torch.ones(motions.shape[0], dtype=int, device=device) * 60)})["mu"]

def get_middle_and_reconstructed_inp(inp):
    inp_list = [ inp ]

    retrieved_motions = retrieve_motions(inp_list, parameters['device'])
    clip_features1 = encode_motions(model, retrieved_motions[:, :, :, :], parameters['device'])
    all_clip_features = [
        clip_features1,
    ]


    all_clip_features = torch.transpose(torch.stack(all_clip_features, axis=0), 0, 1)
    h, w = all_clip_features.shape[:2]
    gendurations = torch.ones((h*w, 1), dtype=int) * parameters['num_frames']

    # generate the repr (joints3D/pose etc)
    model.eval()
    with torch.no_grad():
        generation = model.generate(all_clip_features, gendurations,
                                    is_amass=True,
                                    is_clip_features=True)

    for key, val in generation.items():
        if len(generation[key].shape) == 1:
            generation[key] = val.reshape(h, w)
        else:
            generation[key] = val.reshape(h, w, *val.shape[1:])

    inp_reconstruct = generation["output"][0][0].detach().cpu()

    return all_clip_features[0].detach().cpu(), inp_reconstruct

In [7]:
def select_index(n):
    if n < 60:
        # start with all numbers
        base = np.arange(n)
        # randomly pick extra numbers to fill up to 60
        extra = np.random.choice(base, size=60 - n, replace=True)
        arr = np.concatenate([base, extra])
    else:
        # sample 60 unique numbers
        # arr = np.random.choice(np.arange(n), size=60, replace=False)
        arr = np.arange(60)
    
    return np.sort(arr)

# Example
print(select_index(45))
print(select_index(80))
print(select_index(60))

[ 0  1  2  3  3  4  5  6  7  7  8  8  9 10 10 10 11 12 13 13 14 15 16 17
 17 18 19 20 21 22 23 24 24 24 25 25 26 26 27 28 29 30 31 31 32 33 33 34
 35 36 37 38 39 40 41 41 42 43 43 44]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59]


In [8]:
name_data_list = []
inp_data_list = []
clip_data_list = []
inp_reconstruct_data_list = []
visited = set()

for item_name in tqdm(os.listdir("../output_frames/")):
    data = np.load("../output_frames/%s/motion_sequence.npz" % item_name)

    length = data["smplx_root_pose"].shape[0]
    if length < 10:
        # print("ignore %s with len %d" % (item_name, length))
        continue

    item_name_without_character = item_name[item_name.find("_")+1:]
    if item_name_without_character in visited:
        # print(item_name_without_character)
        continue
    visited.add(item_name_without_character)

    thetas = np.zeros((data["smplx_root_pose"].shape[0], 72), dtype=np.float64)
    thetas[:,  :3 ] = data["smplx_root_pose"]
    thetas[:, 3:66] = data["smplx_body_pose"]
    thetas = torch.from_numpy(thetas)

    sample_indexes = select_index(length)
    thetas_sampled = thetas[sample_indexes].clone()
    inp = theta_to_inp(thetas_sampled, None, False)

    clip_feature, inp_reconstruct = get_middle_and_reconstructed_inp(inp)

    name_data_list.append(item_name)
    inp_data_list.append(inp.numpy())
    clip_data_list.append(clip_feature.numpy())
    inp_reconstruct_data_list.append(inp_reconstruct.numpy())




100%|██████████| 2165/2165 [00:04<00:00, 434.64it/s] 


In [9]:
np.savez("../inpDataset_unique_char.npz", names=name_data_list, clip_arrays=clip_data_list, inp_arrays=inp_data_list, inp_recon_arrays=inp_reconstruct_data_list)


In [10]:
len(name_data_list)

537

In [11]:
import torch
import trimesh
import pyrender

from smplx import SMPL

In [12]:
# Load SMPL model
smpl_model = SMPL(model_path='./models/smpl')

# Example theta
idx = 0
# theta = torch.from_numpy(data["thetas"][0][idx:idx+1]).to(torch.float)
theta = thetas[idx:idx+1].to(torch.float)
output = smpl_model(body_pose=theta[:, 3:], global_orient=theta[:, :3])

vertices = output.vertices.detach().cpu().numpy().squeeze()
faces = smpl_model.faces

# Render using trimesh + pyrender
mesh = trimesh.Trimesh(vertices, faces)
scene = pyrender.Scene()
scene.add(pyrender.Mesh.from_trimesh(mesh))

# # uncomment for change view and get matrix
# viewer = pyrender.Viewer(scene, use_raymond_lighting=True )
# viewer._camera_node.matrix

In [13]:
import torch
from smplx import SMPL
import trimesh
import pyrender
import numpy as np
import imageio
from tqdm import tqdm


def thetas_to_video(motion, output_path):
    # Load SMPL model
    smpl_model = SMPL(model_path='./models/smpl')
    faces = smpl_model.faces

    scene = pyrender.Scene()
    r = pyrender.OffscreenRenderer(640, 480)

    # Add a camera
    camera = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)
    cam_pose = np.array(
            [[ 9.98793959e-01,  1.16659486e-03,  4.90842806e-02, 1.18703522e-01],
            [-7.82186846e-04, -9.99212735e-01,  3.96648358e-02, -1.85050091e-01],
            [ 4.90919110e-02, -3.96553915e-02, -9.98006731e-01, -2.73471684e+00],
            [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 1.00000000e+00]]
    #     [[ 0.43808333, -0.05362655,  0.89733338,  2.46587279],
    #    [-0.19221437, -0.98072038,  0.03523023, -0.04399622],
    #    [ 0.87814386, -0.18791415, -0.43994504, -0.66070501],
    #    [ 0.        ,  0.        ,  0.        ,  1.        ]]
        
    )
    scene.add(camera, pose=cam_pose)

    # Add light
    light = pyrender.DirectionalLight(color=np.ones(3), intensity=3.0)
    scene.add(light, pose=cam_pose)

    # Suppose data["thetas"][0] has shape [T, 72]
    frames = []
    T = len(motion)
    if isinstance(motion, np.ndarray):
        motion = torch.from_numpy(motion)
    motion = motion.to(torch.float)
    for i in tqdm(range(T), desc="Rendering frames"):
        theta = motion[i:i+1]
        output = smpl_model(body_pose=theta[:, 3:], global_orient=theta[:, :3])

        vertices = output.vertices.detach().cpu().numpy().squeeze()
        mesh_visual = pyrender.Mesh.from_trimesh(trimesh.Trimesh(vertices, faces))

        # Clear old mesh, add new one
        scene.clear()
        scene.add(camera, pose=cam_pose)
        scene.add(light, pose=cam_pose)
        scene.add(mesh_visual)

        # Render and save frame
        color, _ = r.render(scene)
        frames.append(color)

    r.delete()


    # Make sure ffmpeg is installed: pip install imageio[ffmpeg]
    with imageio.get_writer(output_path, fps=30, format='ffmpeg') as writer:
        for frame in frames:
            writer.append_data(frame)

    print(f"✅ Saved video as {output_path}")

In [14]:
output_path = "../output/4.smpl_motion_file.mp4"
thetas_to_video(thetas.cpu() , output_path)
Video(output_path, embed=True)

Rendering frames: 100%|██████████| 33/33 [00:00<00:00, 90.30it/s]


✅ Saved video as ../output/4.smpl_motion_file.mp4


In [15]:
print(inp_reconstruct.shape)

torch.Size([25, 6, 60])


In [16]:
thetas_reconstruct = inp_to_theta(inp_reconstruct)
print(thetas_reconstruct.shape)  # (nframes, 72)

output_path = "../output/4.smpl_motion_dataset.mp4"
thetas_to_video(thetas_reconstruct, output_path)

Video(output_path, embed=True)

torch.Size([60, 72])


Rendering frames: 100%|██████████| 60/60 [00:00<00:00, 109.86it/s]


✅ Saved video as ../output/4.smpl_motion_dataset.mp4
